# Predictive Maintenance ML Pipeline

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, f1_score, roc_auc_score, accuracy_score

# Models
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

import warnings
warnings.filterwarnings('ignore')

# Set seaborn style for beautiful plots
sns.set_theme(style='whitegrid')

## 1. Exploratory Data Analysis (EDA)

In [ ]:
data_path = 'ai4i2020.csv'
df = pd.read_csv(data_path)
df.head()

In [ ]:
# 1. Class Imbalance Check
plt.figure(figsize=(6, 4))
sns.countplot(data=df, x='Machine failure', palette='Set2')
plt.title('Distribution of Machine Failures (Target Variable)')
plt.show()

print(f"Failure Rate: {df['Machine failure'].mean() * 100:.2f}%")

In [ ]:
# 2. Feature Distributions by Failure
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.histplot(data=df, x='Torque [Nm]', hue='Machine failure', kde=True, ax=axes[0], palette='Set1')
axes[0].set_title('Torque Distribution by Failure Status')

sns.histplot(data=df, x='Rotational speed [rpm]', hue='Machine failure', kde=True, ax=axes[1], palette='Set1')
axes[1].set_title('Rotational Speed Distribution by Failure Status')
plt.show()

In [ ]:
# 3. Correlation Heatmap (Numeric Features Only)
numeric_cols = df.select_dtypes(include=[np.number]).columns.drop(['UDI', 'Machine failure', 'TWF', 'HDF', 'PWF', 'OSF', 'RNF'])
plt.figure(figsize=(8, 6))
sns.heatmap(df[numeric_cols].corr(), annot=True, cmap='coolwarm', fmt='.2f', linewidths=0.5)
plt.title('Correlation Matrix of Telemetry Data')
plt.show()

## 2. Feature Engineering & Preparation

In [ ]:
# Domain-specific feature engineering
df['temp_diff'] = df['Process temperature [K]'] - df['Air temperature [K]']
df['power_w'] = df['Torque [Nm]'] * (df['Rotational speed [rpm]'] * (2 * np.pi / 60))
df['wear_torque_product'] = df['Tool wear [min]'] * df['Torque [Nm]']

numeric_features = [
    'Air temperature [K]', 'Process temperature [K]', 'Rotational speed [rpm]', 
    'Torque [Nm]', 'Tool wear [min]', 'temp_diff', 'power_w', 'wear_torque_product'
]
categorical_features = ['Type']

X = df[numeric_features + categorical_features]
y = df['Machine failure'].astype(int)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

## 3. Preprocessing Pipeline Setup

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        ('numeric', StandardScaler(), numeric_features),
        ('categorical', OneHotEncoder(handle_unknown='ignore'), categorical_features)
    ]
)

def build_model_pipeline(model):
    return Pipeline(steps=[('preprocessor', preprocessor), ('model', model)])

## 4. Train Models and Compare

In [ ]:
# 1. Baseline Model: Logistic Regression
lr_model = LogisticRegression(class_weight='balanced', random_state=42, max_iter=1000)
lr_pipeline = build_model_pipeline(lr_model)
lr_pipeline.fit(X_train, y_train)

# 2. Random Forest
rf_model = RandomForestClassifier(n_estimators=400, min_samples_leaf=2, class_weight='balanced_subsample', random_state=42, n_jobs=-1)
rf_pipeline = build_model_pipeline(rf_model)
rf_pipeline.fit(X_train, y_train)

# 3. XGBoost
xgb_model = XGBClassifier(n_estimators=400, learning_rate=0.1, max_depth=6, scale_pos_weight=30, random_state=42, n_jobs=-1)
xgb_pipeline = build_model_pipeline(xgb_model)
xgb_pipeline.fit(X_train, y_train)

print('Models trained successfully!')

## 5. Evaluation and Comparison

In [ ]:
pipelines = {
    'Baseline (Logistic Regression)': lr_pipeline,
    'Random Forest': rf_pipeline,
    'XGBoost': xgb_pipeline
}

results = []

for name, pipe in pipelines.items():
    y_pred = pipe.predict(X_test)
    y_proba = pipe.predict_proba(X_test)[:, 1]
    
    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred, average='macro')
    roc_auc = roc_auc_score(y_test, y_proba)
    
    results.append({'Model': name, 'Accuracy': acc, 'Macro F1': f1, 'ROC-AUC': roc_auc})

results_df = pd.DataFrame(results)
results_df.sort_values(by='Macro F1', ascending=False)

## 6. Finalize and Save the Model

Random Forest gives the best balance of Macro F1 and ROC-AUC from the comparison above, so it's the model we ship. We save the **entire pipeline** (preprocessing + model together) as a single `.joblib` file, not just the raw classifier. **Why:** if we only saved the `RandomForestClassifier`, the FastAPI backend would need to manually re-implement the `StandardScaler` and `OneHotEncoder` steps by hand at prediction time — fragile and easy to get subtly wrong. Saving the full pipeline means the backend just loads one file and calls `.predict_proba()` on a raw row; the pipeline handles scaling and encoding internally, exactly the same way it did during training.

In [ ]:
import joblib
joblib.dump(rf_pipeline, 'model.joblib')

#**Next steps (Part 2):**
build a FastAPI backend with Pydantic request validation that loads `model.joblib` and exposes a `/predict` endpoint, then connect a simple frontend, then deploy both so this becomes a live, clickable demo.